# Morrigan SFT v1 — Colab Inference
Model: [JaceSabr/morrigan-sft-v1](https://huggingface.co/JaceSabr/morrigan-sft-v1)

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── Step 1: Install llama-cpp-python with CUDA support ──
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --force-reinstall --no-cache-dir -q

In [ ]:
# ── Step 2: Load the GGUF model ──
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="JaceSabr/morrigan-sft-v1",
    filename="morrigan-Q5_K_M.gguf",
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False,
)
print("Model loaded!")

In [ ]:
# ── Step 3: Check if the GGUF has a chat template ──
template = llm.metadata.get("tokenizer.chat_template", None)
if template:
    print("Chat template found in GGUF:")
    print(template)
else:
    print("⚠️ No chat template in the GGUF — this is why the model was echoing the system prompt.")
    print("We'll format the prompt manually below.")

In [ ]:
# ── Step 4: Define the chat function with manual Llama 3 formatting ──

SYSTEM_PROMPT = "You are Morrigan, a helpful assistant."

def chat(user_message: str, system_prompt: str = SYSTEM_PROMPT) -> str:
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_message}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    output = llm(
        prompt,
        max_tokens=512,
        temperature=0.7,
        stop=["<|eot_id|>", "<|end_of_text|>"],
    )
    return output["choices"][0]["text"].strip()

In [ ]:
# ── Step 5: Test it ──
response = chat("Hello! Who are you?")
print(response)

In [ ]:
# ── Step 6: Try more prompts ──
response = chat("Explain quantum computing in simple terms.")
print(response)

In [ ]:
# ── Bonus: Multi-turn conversation helper ──

def chat_multi(messages: list, system_prompt: str = SYSTEM_PROMPT) -> str:
    """
    Multi-turn chat. Pass a list of dicts:
    [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}, ...]
    """
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt}<|eot_id|>"
    )
    for msg in messages:
        prompt += (
            f"<|start_header_id|>{msg['role']}<|end_header_id|>\n\n"
            f"{msg['content']}<|eot_id|>"
        )
    prompt += "<|start_header_id|>assistant<|end_header_id|>\n\n"

    output = llm(
        prompt,
        max_tokens=512,
        temperature=0.7,
        stop=["<|eot_id|>", "<|end_of_text|>"],
    )
    return output["choices"][0]["text"].strip()


# Example multi-turn
history = [
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a high-level programming language known for its readability and versatility."},
    {"role": "user", "content": "What makes it different from C++?"},
]

response = chat_multi(history)
print(response)